In [1]:
import glob, io, zipfile, gc
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

torch.manual_seed(1)
np.random.seed(1)

In [2]:
DAYS = 90   # raise later if your RAM allows

NEEDED = [
    "FlightDate", "DayOfWeek", "Tail_Number", "CRSDepTime", "CRSElapsedTime",
    "Distance", "DepDel15", "DepDelayMinutes", "ArrDelayMinutes", "Cancelled",
]

frames = []
for zip_path in glob.glob("bts_raw_data/*.zip"):
    with zipfile.ZipFile(zip_path) as archive:
        csv_name = next(name for name in archive.namelist() if name.lower().endswith('.csv'))
        with archive.open(csv_name) as csv_file:
            frames.append(pl.read_csv(io.BytesIO(csv_file.read()), columns=NEEDED))

air_data = pl.concat(frames, how="diagonal_relaxed")
del frames; gc.collect()

last_day = pd.to_datetime(air_data["FlightDate"].cast(pl.Utf8).max())
start_day = (last_day - pd.Timedelta(days=DAYS)).strftime("%Y-%m-%d")
air_data = air_data.filter(pl.col("FlightDate").cast(pl.Utf8) >= start_day)
print(air_data.shape)

(1740778, 10)


In [3]:
flights = air_data.to_pandas()
del air_data; gc.collect()

flights["PredictionTime"] = (
    pd.to_datetime(flights["FlightDate"])
    + pd.to_timedelta((flights["CRSDepTime"] // 100) * 60 + flights["CRSDepTime"] % 100, unit="m")
)
flights = flights.dropna(subset=["PredictionTime", "Tail_Number", "CRSElapsedTime"])
flights = flights.sort_values(["Tail_Number", "PredictionTime"]).reset_index(drop=True)

# when the flight is scheduled to land
flights["SchedLanding"] = flights["PredictionTime"] + pd.to_timedelta(flights["CRSElapsedTime"], unit="m")

by_plane = flights.groupby("Tail_Number")
print(flights.shape)

(1740778, 12)


In [4]:
N_LEGS = 4

def leg_features(k):
    """Numbers describing the plane's k-th previous flight (k=1 is the most recent one)."""
    now = flights["PredictionTime"]
    prev_arr = by_plane["ArrDelayMinutes"].shift(k)
    prev_dep = by_plane["DepDelayMinutes"].shift(k)
    prev_sched_dep = by_plane["PredictionTime"].shift(k)
    prev_land = by_plane["SchedLanding"].shift(k)

    exists = prev_land.notna()
    gap = (now - prev_land).dt.total_seconds() / 60          # minutes since it was scheduled to land
    dep_gap = (now - prev_sched_dep).dt.total_seconds() / 60  # minutes since it was scheduled to leave

    landed = exists & prev_arr.notna() & (prev_arr <= gap)     # had it landed by our departure?
    departed = exists & prev_dep.notna() & (prev_dep <= dep_gap)  # had it left by our departure?

    arr_delay = prev_arr.where(landed, 0).fillna(0).clip(0, 300) / 60
    dep_delay = prev_dep.where(departed, 0).fillna(0).clip(0, 300) / 60
    turnaround = gap.where(exists, 720).clip(0, 720) / 720

    return np.stack([
        arr_delay.to_numpy(dtype="float32"),
        dep_delay.to_numpy(dtype="float32"),
        landed.to_numpy(dtype="float32"),
        departed.to_numpy(dtype="float32"),
        turnaround.to_numpy(dtype="float32"),
    ], axis=1)

# oldest flight first, most recent flight last (the GRU reads them in that order)
X_seq = np.stack([leg_features(k) for k in range(N_LEGS, 0, -1)], axis=1)
print("Sequence array:", X_seq.shape)   # (flights, 4 previous legs, 5 numbers each)

# a few basic facts about the flight itself, scaled to roughly 0-1
X_flat = np.stack([
    flights["CRSDepTime"] // 100 / 24,
    flights["DayOfWeek"] / 7,
    flights["Distance"] / 3000,
    flights["CRSElapsedTime"] / 300,
], axis=1)
X_flat = np.nan_to_num(X_flat).astype("float32")

Sequence array: (1740778, 4, 5)


In [5]:
keep = flights["DepDel15"].notna().to_numpy()       # cancelled flights have no DepDel15

y = flights["DepDel15"].to_numpy()[keep].astype("float32")
X_seq = X_seq[keep]
X_flat = X_flat[keep]
times = flights.loc[keep, "PredictionTime"]

cutoff = times.quantile(0.80)
train_mask = (times < cutoff).to_numpy()

y_test = y[~train_mask]
print("Train:", train_mask.sum(), "| Test:", (~train_mask).sum())
print("Share of flights delayed 15+ min:", y.mean())

Train: 1372069 | Test: 343071
Share of flights delayed 15+ min: 0.22232237


In [6]:
def make_ds(mask):
    return TensorDataset(
        torch.tensor(X_seq[mask]),
        torch.tensor(X_flat[mask]),
        torch.tensor(y[mask]),
    )

train_ds = make_ds(train_mask)
test_ds = make_ds(~train_mask)

train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=8192)

In [7]:
class PropagationNet(nn.Module):
    def __init__(self, leg_features=5, flight_features=4, hidden=32):
        super().__init__()
        self.gru = nn.GRU(leg_features, hidden, batch_first=True)   # reads the plane's last 4 flights
        self.head = nn.Sequential(                                  # combines that with flight facts
            nn.Linear(hidden + flight_features, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, seq, flat):
        _, h = self.gru(seq)            # h = the GRU's summary of the plane's recent history
        h = h[-1]
        return self.head(torch.cat([h, flat], dim=1)).squeeze(1)

model = PropagationNet()
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [8]:
def predict(loader):
    model.eval()
    outputs = []
    with torch.no_grad():
        for seq, flat, _ in loader:
            outputs.append(torch.sigmoid(model(seq, flat)))
    return torch.cat(outputs).numpy()

for epoch in range(1, 6):
    model.train()
    total_loss = 0
    for seq, flat, target in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(seq, flat), target)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(target)

    p = predict(test_loader)
    print(f"epoch {epoch} | train loss {total_loss / len(train_ds):.4f} | test ROC-AUC {roc_auc_score(y_test, p):.3f}")

epoch 1 | train loss 0.4233 | test ROC-AUC 0.804
epoch 2 | train loss 0.3802 | test ROC-AUC 0.810
epoch 3 | train loss 0.3765 | test ROC-AUC 0.814
epoch 4 | train loss 0.3740 | test ROC-AUC 0.818
epoch 5 | train loss 0.3719 | test ROC-AUC 0.819


In [9]:
p = predict(test_loader)
print("Neural net  ROC-AUC:", round(roc_auc_score(y_test, p), 3), "| PR-AUC:", round(average_precision_score(y_test, p), 3))

# baseline: just use "how late did this plane's most recent flight arrive?"
baseline_score = X_seq[~train_mask][:, -1, 0]
print("Baseline (previous arrival delay only) ROC-AUC:", round(roc_auc_score(y_test, baseline_score), 3))
print("Random-guess PR-AUC (base rate):", round(float(y_test.mean()), 3))

Path("models").mkdir(exist_ok=True)
torch.save(model.state_dict(), "models/propagation_net.pt")

Neural net  ROC-AUC: 0.819 | PR-AUC: 0.725
Baseline (previous arrival delay only) ROC-AUC: 0.578
Random-guess PR-AUC (base rate): 0.281


In [10]:
from sklearn.metrics import classification_report
print(classification_report(y_test, (p >= 0.5).astype(int), target_names=["On time", "Delayed 15+"]))

              precision    recall  f1-score   support

     On time       0.81      0.97      0.88    246778
 Delayed 15+       0.84      0.43      0.57     96293

    accuracy                           0.82    343071
   macro avg       0.83      0.70      0.73    343071
weighted avg       0.82      0.82      0.80    343071



In [11]:
from pathlib import Path
Path("sim_data").mkdir(exist_ok=True)

kept = flights.loc[keep, ["Tail_Number", "PredictionTime"]].reset_index(drop=True)
gru_out = kept[~train_mask].copy()
gru_out["gru_prob"] = predict(test_loader)
gru_out.to_csv("sim_data/gru_predictions.csv", index=False)
print(gru_out.shape)

(343071, 3)
